In [ ]:
import json5
import pandas as pd
import regex as re
from Bio import SeqIO
from copy import deepcopy

added = {}
discarded = set()

replaced_accs = set()
replaced_seqs = dict()
replaced_genes = dict()

with open('../sarg.json', 'r') as f:
    for i, j in json5.load(f).items():
        if i != 'discarded':
            for k, l in j.items():
                if k == 'discarded':
                    discarded.update(l)
                if k == 'changed':
                    replaced_seqs.update({z: x for x, y in l.items() for z in y if isinstance(y, list)})
                    replaced_accs.update({z.split('|')[-1] for x, y in l.items() for z in y if isinstance(y, list)})
                    replaced_genes.update({x: y for x, y in l.items() if not isinstance(y, list)})
                if k == 'added':
                    for m, n in l.items():
                        added[m] = 'REF|' + i + '|' + n + '|' + m
        else:
            discarded.update(j)

In [ ]:
import requests

ids = added.keys()

ncbi = []
uniprot = []

for x in ids:
    if x.startswith(("sp|", "tr|")):
        uniprot.append(x.split("|")[1])
    else:
        ncbi.append(x)

with open("../reference/reference.fasta", "w") as out:

    # NCBI
    for i in range(0, len(ncbi), 100):
        batch = ncbi[i:i + 100]

        r = requests.post(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi",
            data={
                "db": "protein",
                "id": ",".join(batch),
                "rettype": "fasta",
                "retmode": "text",
            },
        )
        r.raise_for_status()
        out.write(r.text)

    # UniProt
    for i in range(0, len(uniprot), 100):
        batch = uniprot[i:i + 100]

        query = " OR ".join(
            f"accession:{acc}" for acc in batch
        )

        r = requests.get(
            "https://rest.uniprot.org/uniprotkb/stream",
            params={
                "query": query,
                "format": "fasta",
            },
        )
        r.raise_for_status()
        out.write(r.text)